# 06 — Per-Variant Ensemble Evaluation (Phases 4 + 5)

This notebook evaluates each variant's regime-split ensemble **independently**, answering the paper's core research question per variant:

> *"Does per-regime LSTM training help over a base LSTM at predicting volatility?"*

The research question is answered **three times** — once per pipeline (variants O, A, B) — via Diebold-Mariano tests on `(baseline LSTM)` vs `(regime-split ensemble)` **within the same variant**. Variant H (single LSTM with HMM posterior as feature) is included in the metric table as a reference but has no ensemble of its own, so is not part of the within-variant DM suite.

## Inputs this notebook expects

Artifacts produced upstream by nb 03 (HMMs) + nb 04 (baseline LSTMs) + nb 05 (regime LSTMs):

| Variant | Baseline LSTM | Regime LSTMs | HMM regime probs |
|---|---|---|---|
| O | `lstm_baseline_O.pt` | `lstm_calm_O.pt`, `lstm_volatile_O.pt` | `regime_probabilities_O.parquet` |
| A | `lstm_baseline.pt` | `lstm_calm.pt`, `lstm_volatile.pt` | `regime_probabilities.parquet` |
| H | `lstm_baseline_H.pt` | *(none — architectural variant)* | (uses A's `p_volatile`) |
| B | `lstm_baseline_B.pt` | `lstm_calm_B.pt`, `lstm_volatile_B.pt` | `regime_probabilities_B.parquet` |

## What this notebook produces

1. `data/processed/test_predictions{_O,,_B}.parquet` — per-variant ensemble predictions (via `src/ensemble.py` invocations).
2. `data/processed/test_predictions_H.parquet` — variant H's single-LSTM predictions.
3. Per-variant test-set metric table (MSE / RMSE / MAE / MAPE for baseline + ensemble).
4. **Within-variant DM tests** (O / A / B) — does regime-splitting beat baseline? This is the paper's headline.
5. Per-variant baseline-vs-naive DM (sanity — does the LSTM learn anything beyond persistence?).
6. Regime-stratified breakdown (MSE on strictly-calm vs strictly-volatile subsets, per variant).

The cross-variant comparison (O vs A vs H vs B, plus HAR-RV, plus naive — 15 pairwise tests with multiple-testing correction + F1/F2 diagnostics) happens in **nb 07**, not here.


In [ ]:
import sys
import subprocess
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as sp_stats
import torch

REPO_ROOT = Path().resolve().parent
sys.path.insert(0, str(REPO_ROOT))

import config
from src.ensemble import get_aligned_predictions
from src.utils import regression_metrics

DATA_PROCESSED = config.DATA_PROCESSED
MODELS_DIR = config.MODELS_DIR
LOG_DIR = REPO_ROOT / "logs"
LOG_DIR.mkdir(exist_ok=True)

print("Repo root:", REPO_ROOT)


## Prerequisite check

Confirms all required LSTM checkpoints + HMM regime-probability parquets exist. Missing artifacts here almost always mean a training cell in nb 03 / 04 / 05 was skipped.


In [ ]:
REQUIRED = {
    # Baseline LSTMs — one per variant
    "lstm_baseline_O.pt":    MODELS_DIR / "lstm_baseline_O.pt",
    "lstm_baseline.pt":      MODELS_DIR / "lstm_baseline.pt",
    "lstm_baseline_H.pt":    MODELS_DIR / "lstm_baseline_H.pt",
    "lstm_baseline_B.pt":    MODELS_DIR / "lstm_baseline_B.pt",
    # Regime LSTMs — 3 variants × 2 regimes (H excluded)
    "lstm_calm_O.pt":        MODELS_DIR / "lstm_calm_O.pt",
    "lstm_volatile_O.pt":    MODELS_DIR / "lstm_volatile_O.pt",
    "lstm_calm.pt":          MODELS_DIR / "lstm_calm.pt",
    "lstm_volatile.pt":      MODELS_DIR / "lstm_volatile.pt",
    "lstm_calm_B.pt":        MODELS_DIR / "lstm_calm_B.pt",
    "lstm_volatile_B.pt":    MODELS_DIR / "lstm_volatile_B.pt",
    # HMM artifacts for variants O, A, B
    "regime_probabilities_O.parquet":  DATA_PROCESSED / "regime_probabilities_O.parquet",
    "regime_probabilities.parquet":    DATA_PROCESSED / "regime_probabilities.parquet",
    "regime_probabilities_B.parquet":  DATA_PROCESSED / "regime_probabilities_B.parquet",
    "hmm_meta_O.joblib":     MODELS_DIR / "hmm_meta_O.joblib",
    "hmm_meta.joblib":       MODELS_DIR / "hmm_meta.joblib",
    "hmm_meta_B.joblib":     MODELS_DIR / "hmm_meta_B.joblib",
}
missing = {k: str(v) for k, v in REQUIRED.items() if not v.exists()}
if missing:
    raise RuntimeError(
        f"Missing upstream artifacts:\n" +
        "\n".join(f"  {k}: {v}" for k, v in missing.items()) +
        "\n\nRun nb 03 → nb 04 → nb 05 on Colab to produce these."
    )
print(f"All {len(REQUIRED)} upstream artifacts present.")


## 1. Produce ensemble predictions per variant

Three invocations of `src/ensemble.py`, one per ensemble variant (O, A, B). Variant H is a single baseline LSTM — no ensemble step; predictions come directly from `get_aligned_predictions`.


In [ ]:
ENSEMBLE_VARIANTS = [
    {
        "name":        "O",
        "suffix":      "_O",
        "regime_probs": DATA_PROCESSED / "regime_probabilities_O.parquet",
        "hmm_meta":    MODELS_DIR / "hmm_meta_O.joblib",
    },
    {
        "name":        "A",
        "suffix":      "",
        "regime_probs": DATA_PROCESSED / "regime_probabilities.parquet",
        "hmm_meta":    MODELS_DIR / "hmm_meta.joblib",
    },
    {
        "name":        "B",
        "suffix":      "_B",
        "regime_probs": DATA_PROCESSED / "regime_probabilities_B.parquet",
        "hmm_meta":    MODELS_DIR / "hmm_meta_B.joblib",
    },
]

for v in ENSEMBLE_VARIANTS:
    name, suffix = v["name"], v["suffix"]
    cmd = [
        sys.executable, "-u", "-m", "src.ensemble",
        "--variant-suffix", suffix,
        "--regime-probs-path", str(v["regime_probs"]),
        "--hmm-meta-path", str(v["hmm_meta"]),
    ]
    print(f"[Variant {name}] running: {' '.join(cmd)}")
    log_path = LOG_DIR / f"ensemble_{name or 'A'}.log"
    with open(log_path, "w") as log_f:
        result = subprocess.run(cmd, cwd=str(REPO_ROOT), stdout=log_f, stderr=subprocess.STDOUT)
    if result.returncode != 0:
        print(f"  FAILED (rc={result.returncode}) — see {log_path}")
    else:
        out_parquet = DATA_PROCESSED / f"test_predictions{suffix}.parquet"
        print(f"  ok → {out_parquet.name} (log: {log_path.name})")


In [ ]:
# Variant H — single-LSTM predictions, no ensemble step.
device = torch.device("cpu")
test_df = pd.read_parquet(DATA_PROCESSED / "test.parquet")

pred_H = get_aligned_predictions("baseline", test_df, device, suffix="_H")
pred_H.name = "baseline"
df_H = pred_H.to_frame()
df_H["target"] = test_df[config.LSTM_TARGET]
df_H = df_H.dropna()
out_path_H = DATA_PROCESSED / "test_predictions_H.parquet"
df_H.to_parquet(out_path_H)
print(f"Variant H: saved {len(df_H)} predictions → {out_path_H.name}")


## 2. Load all prediction parquets and build per-variant predictor dict

Each variant's baseline and (if applicable) ensemble columns are loaded from its respective parquet. Every comparison in the cells below is done **within a variant** on its own index to get the natural row count — we do not intersect across variants here (that's nb 07's job).


In [ ]:
preds = {}

for v in ENSEMBLE_VARIANTS:
    name, suffix = v["name"], v["suffix"]
    p = pd.read_parquet(DATA_PROCESSED / f"test_predictions{suffix}.parquet")
    # Inside each variant's parquet, baseline/calm/volatile/ensemble columns exist
    preds[name] = {
        "baseline":  p["baseline"].values,
        "calm":      p["calm"].values,
        "volatile":  p["volatile"].values,
        "ensemble":  p["ensemble"].values,
        "target":    p["target"].values,
        "p_calm":    p["p_calm"].values,
        "p_volatile": p["p_volatile"].values,
        "index":     p.index,
    }
    print(f"  {name}: {len(p)} rows  ({p.index.min().date()} → {p.index.max().date()})")

# Variant H
pH = pd.read_parquet(DATA_PROCESSED / "test_predictions_H.parquet")
preds["H"] = {
    "baseline":  pH["baseline"].values,
    "target":    pH["target"].values,
    "index":     pH.index,
}
print(f"  H: {len(pH)} rows  ({pH.index.min().date()} → {pH.index.max().date()})")

# Naive baseline (rolling_std_21) comes from the test_df — will be reindexed per variant.
naive_full = test_df["rolling_std_21"].dropna()
print(f"  naive (rolling_std_21): {len(naive_full)} rows available")


## 3. Per-variant test-set metric tables

One table per variant. Rows: naive, baseline, calm (ensemble component, if present), volatile (component), ensemble. Columns: MSE / RMSE / MAE / MAPE.


In [ ]:
def _variant_metrics_table(name):
    p = preds[name]
    y_true = p["target"]
    naive_aligned = naive_full.reindex(p["index"]).values

    rows = [("naive", naive_aligned), ("baseline", p["baseline"])]
    if "ensemble" in p:
        rows += [("calm", p["calm"]), ("volatile", p["volatile"]), ("ensemble", p["ensemble"])]

    records = []
    for label, pred in rows:
        m = regression_metrics(y_true, pred)
        records.append({
            "model": label,
            "MSE":   m["MSE"],
            "RMSE":  m["RMSE"],
            "MAE":   m["MAE"],
            "MAPE":  m.get("MAPE"),
        })
    return pd.DataFrame(records).set_index("model")

for vname in ["O", "A", "H", "B"]:
    print(f"\n── Variant {vname} (n = {len(preds[vname]['target'])}) ──")
    display(_variant_metrics_table(vname))


## 4. WITHIN-VARIANT Diebold-Mariano tests — the research question

**The paper's headline comparison.** For each of variants O / A / B we test whether the regime-split ensemble significantly beats the baseline LSTM within the same pipeline, on both MSE and MAE loss. Positive DM ⇒ first-listed predictor has higher loss ⇒ second-listed wins.

Variant H is excluded — it has no ensemble; its role is tested in nb 07 cross-variant.

h = 21 (target horizon), Bartlett HAC kernel, HLN small-sample correction.


In [ ]:
def diebold_mariano(y_true, y_a, y_b, h=21, loss="mse"):
    y_true = np.asarray(y_true, float); y_a = np.asarray(y_a, float); y_b = np.asarray(y_b, float)
    e_a = (y_a - y_true)**2 if loss == "mse" else np.abs(y_a - y_true)
    e_b = (y_b - y_true)**2 if loss == "mse" else np.abs(y_b - y_true)
    d = e_a - e_b
    n = len(d); d_bar = float(d.mean())
    max_lag = max(h - 1, 0)
    gamma0 = float(np.var(d, ddof=0)); S = gamma0
    for k in range(1, max_lag + 1):
        w = 1.0 - k / (max_lag + 1)
        gamma_k = float(np.mean((d[k:] - d_bar) * (d[:-k] - d_bar)))
        S += 2.0 * w * gamma_k
    if S <= 0: S = gamma0
    dm_raw = d_bar / np.sqrt(S / n)
    hln = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    dm = float(dm_raw * hln)
    p  = float(2.0 * (1.0 - sp_stats.t.cdf(np.abs(dm), df=n - 1)))
    return {"dm_stat": dm, "p_value": p, "n": n}


def _sig_flag(p, alpha=0.05):
    if p < 0.01:  return "***"
    if p < 0.05:  return "**"
    if p < 0.10:  return "*"
    return "n.s."


rows = []
for vname in ["O", "A", "B"]:
    p = preds[vname]
    y_true = p["target"]
    # DM(baseline, ensemble) — does regime-splitting HELP within this variant?
    for loss in ["mse", "mae"]:
        r = diebold_mariano(y_true, p["baseline"], p["ensemble"], loss=loss)
        sig = _sig_flag(r["p_value"])
        verdict = ("ensemble wins" if r["dm_stat"] > 0 else "baseline wins") if r["p_value"] < 0.05 else "tie"
        rows.append({
            "variant":      vname,
            "comparison":   "baseline vs ensemble",
            "loss":         loss.upper(),
            "DM":           round(r["dm_stat"], 3),
            "p_value":      round(r["p_value"], 4),
            "significance": sig,
            "verdict":      verdict,
            "n":            r["n"],
        })

within_variant_dm = pd.DataFrame(rows)
print("Within-variant DM (baseline vs ensemble):")
within_variant_dm


## 5. Each predictor vs naive (DM sanity)

Does each variant's LSTM learn **anything** beyond 21-day persistence? This is the floor check — if a variant's baseline or ensemble doesn't beat naive, that variant is a pure persistence-echo at this horizon (per the F1 finding).


In [ ]:
rows = []
for vname in ["O", "A", "H", "B"]:
    p = preds[vname]
    y_true = p["target"]
    naive_aligned = naive_full.reindex(p["index"]).values

    # Every variant has a baseline; only O/A/B have ensembles.
    predictors = [("baseline", p["baseline"])]
    if "ensemble" in p:
        predictors.append(("ensemble", p["ensemble"]))

    for label, pred in predictors:
        for loss in ["mse", "mae"]:
            r = diebold_mariano(y_true, naive_aligned, pred, loss=loss)
            sig = _sig_flag(r["p_value"])
            verdict = (f"{label} wins" if r["dm_stat"] > 0 else "naive wins") if r["p_value"] < 0.05 else "tie"
            rows.append({
                "variant": vname, "predictor": label, "loss": loss.upper(),
                "DM": round(r["dm_stat"], 3), "p_value": round(r["p_value"], 4),
                "significance": sig, "verdict": verdict, "n": r["n"],
            })

naive_dm = pd.DataFrame(rows)
print("Each predictor vs naive:")
naive_dm


## 6. Regime-stratified per-variant breakdown

For each variant, split its test rows by its own HMM's `p_volatile > 0.8` threshold. Reports MSE for baseline + ensemble on strictly-calm vs strictly-volatile subsets. The volatile subset is typically tiny (~3-5 % base rate × test window), so per-subset DM isn't run — point estimates only.

Per the F2 finding, we expect the volatile regime's LSTM to produce near-constant output (degenerate on ~180 training windows). The subset MSE shows whether the ensemble nevertheless recovers some signal on volatile days over the baseline.


In [ ]:
rows = []
for vname in ["O", "A", "B"]:
    p = preds[vname]
    y_true = p["target"]
    is_strict_volatile = p["p_volatile"] > 0.8

    for subset_name, mask in [("strictly calm (p_cal>0.8)", ~is_strict_volatile), ("strictly volatile (p_vol>0.8)", is_strict_volatile)]:
        if mask.sum() == 0:
            continue
        for label, pred_arr in [("baseline", p["baseline"]), ("ensemble", p["ensemble"])]:
            m = regression_metrics(y_true[mask], pred_arr[mask])
            rows.append({
                "variant":   vname,
                "subset":    subset_name,
                "n":         int(mask.sum()),
                "model":     label,
                "MSE":       m["MSE"],
                "MAE":       m["MAE"],
            })

regime_breakdown = pd.DataFrame(rows).set_index(["variant", "subset", "model"])
print("Regime-stratified MSE / MAE per variant:")
regime_breakdown


## 7. Summary (fill in after execution)

Fill these in from the tables above once the notebook has run on real trained checkpoints:

### Core research-question answer — within-variant DM (§4)

| Variant | DM MSE (p) | DM MAE (p) | Verdict | Interpretation |
|---|---|---|---|---|
| O (stationary only) | — | — | — | does regime-splitting help in the pre-everything pipeline? |
| A (stationary + sentiment) | — | — | — | does regime-splitting help once sentiment is in? |
| B (+ VIX family) | — | — | — | does regime-splitting help once forward-looking IV is in? |

If all three → **tie**, this is the paper's structural-limits headline: regime-splitting doesn't help at h=21 on daily price/volume data regardless of feature richness (consistent with F1, F2, F3, F4).

If one or more → **ensemble wins**, quantify the effect size and discuss which feature-richness regime it appears in.

If one or more → **baseline wins** (regime-splitting actively *hurts*), this is also a clean finding — the ensemble's constant-volatile-LSTM output (F2) drags calm-day predictions up.

### Supplementary observations

- **Predictor vs naive** (§5): if any variant's baseline fails to reject naive on MAE, that variant is a pure persistence-echo. Expected: variants A and B should reject with MAE p < 0.01; variant O is the interesting test.
- **Regime-stratified breakdown** (§6): look for the pattern "ensemble wins on strictly-volatile subset but loses on strictly-calm" — the regime-split working exactly where the HMM gates to the volatile LSTM, but hurt by calm contamination when `p_volatile > 0` slightly.

### Hand-off to nb 07

nb 06 is the per-variant deep-dive. nb 07 (variant-comparison notebook) does the **cross-variant** 15-pair DM matrix + Bonferroni + BH-FDR correction + F1 cross-correlation diagnostic + F2 prediction-std diagnostic, including HAR-RV as an econometric reference.
